In [185]:
# This code extracts the sequence and chain informations according to the assigned tutorials  

from biopandas.pdb import PandasPdb
import numpy as np

import os

# Specify the directory where you want to save the files
output_directory = "../Outputs/data_manipulation"

# Create the directory if it doesn't exist
if not os.path.exists(output_directory):
    os.makedirs(output_directory)

In [186]:
# Initialize a new PandasPdb object and fetch the PDB file from rcsb.org
# ppdb = PandasPdb().fetch_pdb('1crn')
# ppdb = PandasPdb().fetch_pdb('2C0K')
ppdb = PandasPdb().fetch_pdb('1L2Y')

# Display the type of information in each dataframe
for df_name in ppdb.df:
    print(f"Dataframe: {df_name}")
    print(ppdb.df[df_name].info())
    print(ppdb.df[df_name].head(), "\n")

# Optionally, display the column names for each dataframe
for df_name in ppdb.df:
    print(f"Columns in {df_name}:")
    print(ppdb.df[df_name].columns, "\n")

Dataframe: ATOM
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11552 entries, 0 to 11551
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   record_name     11552 non-null  object 
 1   atom_number     11552 non-null  int64  
 2   blank_1         11552 non-null  object 
 3   atom_name       11552 non-null  object 
 4   alt_loc         11552 non-null  object 
 5   residue_name    11552 non-null  object 
 6   blank_2         11552 non-null  object 
 7   chain_id        11552 non-null  object 
 8   residue_number  11552 non-null  int64  
 9   insertion       11552 non-null  object 
 10  blank_3         11552 non-null  object 
 11  x_coord         11552 non-null  float64
 12  y_coord         11552 non-null  float64
 13  z_coord         11552 non-null  float64
 14  occupancy       11552 non-null  float64
 15  b_factor        11552 non-null  float64
 16  blank_4         11552 non-null  object 
 17  segment_id     

In [187]:
df_atoms = ppdb.df["ATOM"]
df_atoms

,record_name,atom_number,blank_1,atom_name,alt_loc,residue_name,blank_2,chain_id,residue_number,insertion,...,x_coord,y_coord,z_coord,occupancy,b_factor,blank_4,segment_id,element_symbol,charge,line_idx
0,ATOM,1,,N,,ASN,,A,1,,...,-8.901,4.127,-0.555,1.0,0.0,,,N,NaN,175
1,ATOM,2,,CA,,ASN,,A,1,,...,-8.608,3.135,-1.618,1.0,0.0,,,C,NaN,176
2,ATOM,3,,C,,ASN,,A,1,,...,-7.117,2.964,-1.897,1.0,0.0,,,C,NaN,177
3,ATOM,4,,O,,ASN,,A,1,,...,-6.634,1.849,-1.758,1.0,0.0,,,O,NaN,178
4,ATOM,5,,CB,,ASN,,A,1,,...,-9.437,3.396,-2.889,1.0,0.0,,,C,NaN,179
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11547,ATOM,300,,H,,SER,,A,20,,...,1.377,8.948,0.073,1.0,0.0,,,H,NaN,11833
11548,ATOM,301,,HA,,SER,,A,20,,...,-0.962,9.316,1.705,1.0,0.0,,,H,NaN,11834
11549,ATOM,302,,HB2,,SER,,A,20,,...,0.282,10.819,3.316,1.0,0.0,,,H,NaN,11835
11550,ATOM,303,,HB3,,SER,,A,20,,...,0.461,9.076,3.584,1.0,0.0,,,H,NaN,11836


In [188]:
df_others = ppdb.df["OTHERS"]
df_others

,record_name,entry,line_idx
0,HEADER,DE NOVO PROTEIN 25...,0
1,TITLE,NMR STRUCTURE OF TRP-CAGE MINIPROTEIN CONS...,1
2,COMPND,MOL_ID: 1;,2
3,COMPND,2 MOLECULE: TC5B;,3
4,COMPND,3 CHAIN: A;,4
...,...,...,...
285,MODEL,38,11533
286,TER,305 SER A 20,11838
287,ENDMDL,,11839
288,MASTER,136 0 0 2 0 0 0 6 ...,11840


TUTORIAL 1:
Extract sequence from a PDB ATOM record and save to a fasta file format

In [189]:
# Extract the HEADER part of the PDB file
header_entry = df_others[df_others['record_name'] == 'HEADER']['entry'].values
print(header_entry)

# Pull the last string. Corresponds to the code of the protein
header_entry = header_entry[0]
code_name = header_entry.split()[-1]

print("\nCode:",code_name)

['    DE NOVO PROTEIN                         25-FEB-02   1L2Y']

Code: 1L2Y


In [190]:
# Extract the COMPOUND part of the PDB file
compnd_entry = df_others[df_others['record_name'] == 'COMPND']['entry'].values

print(compnd_entry)

molecule_name = None
chains = None

# Iterate over each line in the array
# This section of code assumes there is only one set of chain and one sequence in the protein
# For multiple set of chains and sequences, this code can be modified by adding loops
for line in compnd_entry:
    # Check if the line contains the molecule name
    if 'MOLECULE' in line:
        molecule_name = line.split(':')[1].strip()
        molecule_name = molecule_name.replace(";","")
    # Check if the line contains chain information
    if 'CHAIN' in line:
        chains = (line.split(':')[1].strip())
        chains = chains.replace(";","")

print("\nMolecule Name: ", molecule_name)
print("Chains: ", chains)


['    MOL_ID: 1;' '   2 MOLECULE: TC5B;' '   3 CHAIN: A;'
 '   4 ENGINEERED: YES']

Molecule Name:  TC5B
Chains:  A


In [199]:
# Extract the SOURCE part of the PDB file
source_entry = df_others[df_others['record_name'] == 'SOURCE']['entry'].values
print(source_entry)

scientific_name = ""
taxid_name = ""

# Iterate over each line in the array
for line in source_entry:
    # Check if the line contains the scientific name
    if 'ORGANISM_SCIENTIFIC' in line:
        scientific_name = line.split(':')[1].strip()
        scientific_name = scientific_name.replace(";","")
    if 'ORGANISM_TAXID' in line:
        taxid_name = line.split(':')[1].strip()
        taxid_name = taxid_name.replace(";","")
        taxid_name = "(" + taxid_name + ")" 

print(f"\nOrganism Name: {scientific_name}")
print("Tax ID:" + taxid_name)

['    MOL_ID: 1;' '   2 SYNTHETIC: YES;'
 '   3 OTHER_DETAILS: THE PROTEIN WAS SYNTHESIZED USING STANDARD FMOC'
 '   4 SOLID-PHASE SYNTHESIS METHODS ON AN APPLIED BIOSYSTEMS 433A PEPTIDE'
 '   5 SYNTHESIZER.']

Organism Name: 
Tax ID:


In [192]:
# Extract each residue name and convert them to one letter abbreviations of amino acids
amino_acid_sequence = ppdb.amino3to1().residue_name

# print("Amino Acid Sequence: ")
# print(''.join(amino_acid_sequence))

amino_acid_sequence2 = ppdb.amino3to1()
print(amino_acid_sequence2)

amino_acid_sequence = ''.join(amino_acid_sequence)
print(amino_acid_sequence)

      chain_id residue_name
0            A            N
16           A            L
35           A            Y
56           A            I
75           A            Q
...        ...          ...
11474        A            R
11498        A            P
11512        A            P
11526        A            P
11540        A            S

[760 rows x 2 columns]
NLYIQWLKDGGPSSGRPPPSNLYIQWLKDGGPSSGRPPPSNLYIQWLKDGGPSSGRPPPSNLYIQWLKDGGPSSGRPPPSNLYIQWLKDGGPSSGRPPPSNLYIQWLKDGGPSSGRPPPSNLYIQWLKDGGPSSGRPPPSNLYIQWLKDGGPSSGRPPPSNLYIQWLKDGGPSSGRPPPSNLYIQWLKDGGPSSGRPPPSNLYIQWLKDGGPSSGRPPPSNLYIQWLKDGGPSSGRPPPSNLYIQWLKDGGPSSGRPPPSNLYIQWLKDGGPSSGRPPPSNLYIQWLKDGGPSSGRPPPSNLYIQWLKDGGPSSGRPPPSNLYIQWLKDGGPSSGRPPPSNLYIQWLKDGGPSSGRPPPSNLYIQWLKDGGPSSGRPPPSNLYIQWLKDGGPSSGRPPPSNLYIQWLKDGGPSSGRPPPSNLYIQWLKDGGPSSGRPPPSNLYIQWLKDGGPSSGRPPPSNLYIQWLKDGGPSSGRPPPSNLYIQWLKDGGPSSGRPPPSNLYIQWLKDGGPSSGRPPPSNLYIQWLKDGGPSSGRPPPSNLYIQWLKDGGPSSGRPPPSNLYIQWLKDGGPSSGRPPPSNLYIQWLKDGGPSSGRPPPSNLYIQWLKDGGPSSGRPPPSNLYIQWLKDGGPSSGRPPPS

In [193]:
# Order:
# code_name
# chains
# molecule_name
# scientific_name
# taxid_name
# amino_acid_sequence

first_line = code_name + "|" + "Chains " + chains + "|" + molecule_name + "|" + scientific_name + " " + taxid_name
second_line = amino_acid_sequence

In [194]:
# Write sequences to a FASTA file

file_name = os.path.join(output_directory, f"tutorial_1.fasta")

with open(file_name, "w") as fasta_file:
    fasta_file.write(f">{first_line}\n")
    fasta_file.write(f"{second_line}\n")

TUTORIAL 2:
Extract the full sequence from the SEQRES record and save it to a fasta file format. 

In [195]:
seqres_entry = df_others[df_others['record_name'] == 'SEQRES']['entry'].values
seqres_entry

array(['   1 A   20  ASN LEU TYR ILE GLN TRP LEU LYS ASP GLY GLY PRO SER',
       '   2 A   20  SER GLY ARG PRO PRO PRO SER'], dtype=object)

In [196]:
# Initialize an empty dictionary
result_dict = {}

# Process each string in the array
for item in seqres_entry:
    # Split the string into words
    words = item.split()
    
    # Extract the second letter to get the chain identifier "A"
    second_letter = words[1]
    
    # Extract the substring after the '46', which starts from the 4th word
    substring = ' '.join(words[3:])
    
    # Append the substring to the corresponding key in the dictionary
    if second_letter in result_dict:
        result_dict[second_letter].append(substring)
    else:
        result_dict[second_letter] = [substring]

print(result_dict)

# Initialize a new dictionary to store the merged strings
merged_dict = {}

# Iterate through each key in the dictionary
for key, strings in result_dict.items():
    # Merge all strings in the list into one string
    merged_string = ' '.join(strings)
    # Add the merged string to the new dictionary
    merged_dict[key] = merged_string

print(merged_dict)

{'A': ['ASN LEU TYR ILE GLN TRP LEU LYS ASP GLY GLY PRO SER', 'SER GLY ARG PRO PRO PRO SER']}
{'A': 'ASN LEU TYR ILE GLN TRP LEU LYS ASP GLY GLY PRO SER SER GLY ARG PRO PRO PRO SER'}


In [197]:
# Mapping dictionary from three-letter to one-letter amino acid codes
three_to_one = {
    "ALA": "A", "ARG": "R", "ASN": "N", "ASP": "D", "CYS": "C",
    "GLU": "E", "GLN": "Q", "GLY": "G", "HIS": "H", "ILE": "I",
    "LEU": "L", "LYS": "K", "MET": "M", "PHE": "F", "PRO": "P",
    "SER": "S", "THR": "T", "TRP": "W", "TYR": "Y", "VAL": "V", "SEC": "U"
}

# Function to convert three-letter codes to one-letter codes
def convert_to_one_letter(amino_acid_sequence):
    one_letter_sequence = ''.join(three_to_one[aa] for aa in amino_acid_sequence.split())
    return one_letter_sequence

# Convert the sequences in the dictionary
converted_amino_acids = {key: convert_to_one_letter(seq) for key, seq in merged_dict.items()}

# Print the converted sequences
for key, seq in converted_amino_acids.items():
    print(f"{key}: {seq}")

# Store the converted sequences in strings
sequence_A = converted_amino_acids['A']
sequence_B = converted_amino_acids['B']

merged_sequence = "" 

for chain in converted_amino_acids:
    merged_sequence += converted_amino_acids[chain]

print(merged_sequence)    

A: NLYIQWLKDGGPSSGRPPPS


KeyError: 'B'

In [ ]:
file_name2 = os.path.join(output_directory, f"tutorial_2.fasta")

# Save the the all sequence at once
with open(file_name2, "w") as fasta_file:
    fasta_file.write(f">")
    fasta_file.write(f"{merged_sequence} ")

TUTORIAL 3:
Pick a PDB with multiple chains. Extract chains and save each chain to an individual PDB.

In [ ]:
sequence = ppdb.amino3to1()
print(sequence)

for chain_id in sequence['chain_id'].unique():
    print('\nChain ID: %s' % chain_id)
    print(''.join(sequence.loc[sequence['chain_id'] == chain_id, 'residue_name']))
    
    sequence_by_chain = ''.join(sequence.loc[sequence['chain_id'] == chain_id, 'residue_name'])

    file_name3 = os.path.join(output_directory, f"tutorial_3_chain_{chain_id}.pdb")
    with open(file_name3, 'w') as pdb_file:
        pdb_file.write(f"Chain {chain_id}: \n{sequence_by_chain}")


     chain_id residue_name
0           A            A
5           A            T
12          A            L
20          A            P
27          A            Q
...       ...          ...
4435        E            D
4443        E            K
4452        E            D
4460        E            D
4468        E            T

[550 rows x 2 columns]

Chain ID: A
ATLPQLTPTLVSLLEVIEPEVLYAGYDSSVPDSTWRIMTTLNMLGGRQVIAAVKWAKAIPGFRNLHLDDQMTLLQYSWMSLMAFALGWRSYRQSSANLLCFAPDLIINEQRMTLPCMYDQCKHMLYVSSELHRLQVSYEEYLCMKTLLLLSSVPKDGLKSQELFDEIRMTYIKELGKAIVKREGNSSQNWQRFYQLTKLLDSMHEVVENLLNYCFQTFLDKTMSIEFPEMLAEIITNQIPKYSNGNIKKLLFHQK

Chain ID: B
PVSPKKKENALLRYLLDKDDT

Chain ID: D
LPQLTPTLVSLLEVIEPEVLYAGYDSSVPDSTWRIMTTLNMLGGRQVIAAVKWAKAIPGFRNLHLDDQMTLLQYSWMSLMAFALGWRSYRQSSANLLCFAPDLIINEQRMTLPCMYDQCKHMLYVSSELHRLQVSYEEYLCMKTLLLLSSVPKDGLKSQELFDEIRMTYIKELGKAIVKREGNSSQNWQRFYQLTKLLDSMHEVVENLLNYCFQTFLDKTMSIEFPEMLAEIITNQIPKYSNGNIKKLLFHQK

Chain ID: E
PVSPKKKENALLRYLLDKDDT
